# Descargar los datos

Descarga de la DIAN los archivos mensuales de importaciones y exportaciones, y sus totales de control, en `data/raw/`.

## De dónde salen

La DIAN publica las bases estadísticas de comercio exterior en [esta página](https://www.dian.gov.co/dian/cifras/Paginas/Bases-Estadisticas-de-Comercio-Exterior-Importaciones-y-Exportaciones.aspx) (copia en `docs/`). Los archivos están en dos bibliotecas de documentos de SharePoint:

- Importaciones: `https://www.dian.gov.co/dian/cifras/Basesestadisticasimportaciones/`
- Exportaciones: `https://www.dian.gov.co/dian/cifras/Basesestadisticasexportaciones/`

Los enlaces de la página a esas bibliotecas piden iniciar sesión, pero su contenido se puede listar sin login con la API REST pública de SharePoint (`/dian/cifras/_api/web/GetFolderByServerRelativeUrl('<biblioteca>')/Files`), que devuelve el nombre, el tamaño y la fecha de modificación de cada archivo. Cada archivo se descarga directamente desde su URL en la biblioteca.

Cada biblioteca tiene:

- **Un zip por mes**, desde enero de 2001, con un único `.xlsx` adentro: `07_Importaciones_2026_Julio.zip`, `08_Exportaciones_2026_Agosto.zip`, etc.
- **Un zip de totales de control** (`Totales-de-control-Importaciones-2026.zip`, `Totales-Exportaciones-2026.zip`): para cada mes desde 2001, la cantidad de registros, el valor FOB (y CIF en importaciones) y el peso neto. Se usan en `0.1` para validar cada mes.

## Cómo funciona la descarga

- **Solo baja lo que falta.** Un archivo que ya está en `data/raw/` con el mismo tamaño y la misma fecha de modificación que en el servidor no se vuelve a bajar. La fecha del servidor se copia al archivo descargado para poder hacer esta comparación en corridas posteriores.
- **Se puede retomar.** La descarga completa son unos 63 GB y el servidor de la DIAN entrega alrededor de 0,5 MB/s (unas 30 horas). Si se interrumpe (se corta la conexión, se apaga la computadora o se detiene el notebook), alcanza con volver a correr esta celda:
  - los archivos ya completos se saltean;
  - el archivo que quedó a medias sigue desde el último byte recibido (el servidor de la DIAN acepta pedidos HTTP `Range`). Mientras tanto se guarda en `data/interim/descargas_parciales/`, nunca en `data/raw/`;
  - si se corta la conexión durante un archivo, se reintenta automáticamente hasta 5 veces, con esperas crecientes;
  - si un archivo falla igual, la descarga sigue con los demás y al final lista los que fallaron.
- **Se verifica cada archivo.** Antes de pasarlo a `data/raw/` se comprueba que tenga el tamaño que informa el servidor y que sea un zip válido.
- **`data/raw/` nunca se sobrescribe.** La DIAN corrige meses ya publicados y vuelve a subir los archivos (por ejemplo, al publicar agosto de 2026 también volvió a subir junio y julio). La versión nueva se guarda al lado de la anterior como `<nombre>_<fecha del servidor>.zip`, y `0.1` usa siempre la más reciente.
- **Pausa entre pedidos.** Se espera 2 segundos entre archivos para no sobrecargar el servidor.

Los archivos quedan en `data/raw/<flujo>/<año>/` y los totales de control en `data/raw/<flujo>/totales/`.

## Desde la terminal

Lo mismo se puede correr sin abrir Jupyter, lo que conviene para una descarga de muchas horas (un notebook se detiene si se cierra el editor):

```
uv run colombia-dian-download
uv run colombia-dian-download --years 2024 2025 --flujo exportaciones
```

Se interrumpe con `Ctrl+C` y se retoma corriendo el mismo comando. `--help` muestra todas las opciones.

El código está en `colombia_dian_comex/pipeline.py` (`list_remote_files`, `download_file`, `download_all`) y `colombia_dian_comex/download.py` (`download`), compartido con la actualización automática.

## Configuración

`YEARS` limita la descarga a ciertos años (por ejemplo `[2024, 2025]`), útil para probar el proceso o bajar los datos por partes. Con `None` se descarga todo, desde 2001. Los totales de control se descargan siempre.

In [ ]:
import logging

from colombia_dian_comex.download import download
from colombia_dian_comex.pipeline import Flow

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

YEARS: list[int] | None = None

In [ ]:
# Lists the files that failed in each flow (empty if everything was downloaded).
download(list(Flow), YEARS)